Groundwater | Transport Modeling Track

# Step 3: From Perceptual to Conceptual — MODFLOW 6 for Heat Transport

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → **3-MODFLOW** → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 2, you built a perceptual model of heat transport in the Limmat Valley. You estimated a seepage velocity of ~11 m/d, a thermal retardation factor of ~3.5, and identified the Limmat River (12.5 °C) as the dominant thermal input — 2 °C warmer than background groundwater. Now we translate these insights into the language of MODFLOW 6: equations, numerical stability criteria, and transport packages.

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~25–35 min core + 10 min optional sections |
| **Prerequisites** | Flow track complete, Transport Steps 1–2 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Write the **advection-dispersion equation** for heat and explain each term
2. Evaluate **grid Peclet and Courant numbers** for transport stability
3. Identify **MODFLOW 6 GWE packages** and their roles
4. Map **perceptual transport features** to specific GWE package inputs

In [ ]:
# Setup
import sys

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from shared_functions import check_task_with_solution, create_multiple_choice

---

## Introduction

In the flow track (Step 3), you learned how MODFLOW 6 discretises the **flow equation** — control volumes, hydraulic conductivity averaging, grid types, and boundary condition packages. All of that carries over unchanged to a transport simulation: **the same grid, the same flow solution, the same boundary conditions drive transport.**

What's new for transport?

| Already covered (flow NB3) | New in this notebook |
|---|---|
| Governing flow equation | Advection-dispersion equation (ADE) |
| CVFD discretisation, K-averaging | Numerical stability (Peclet, Courant) |
| Grid types (DIS, DISV, DISU) | Transport-specific packages (GWE) |
| GWF packages (RIV, RCH, WEL, …) | How GWF fluxes carry temperature (SSM) |

> 💡 **Why MODFLOW 6 for transport?** The Groundwater Energy (GWE) and Groundwater Transport (GWT) models are fully integrated into MODFLOW 6. They share the same grid as the Groundwater Flow (GWF) model, run in the same simulation, and are fully scriptable via FloPy — no separate transport code needed.

---

## 1 — The Advection-Dispersion Equation

### General form

The **advection-dispersion equation (ADE)** describes how a dissolved substance (or heat) is transported through a porous medium:

$$\frac{\partial (n C)}{\partial t} = \nabla \cdot (n \mathbf{D} \nabla C) - \nabla \cdot (n \mathbf{v} C) + q_s$$

| Term | Name | Physical meaning |
|---|---|---|
| $\partial(nC)/\partial t$ | **Storage** | Change in mass (or energy) stored per unit volume |
| $\nabla \cdot (n \mathbf{D} \nabla C)$ | **Dispersion** | Spreading due to velocity variations + molecular diffusion |
| $\nabla \cdot (n \mathbf{v} C)$ | **Advection** | Bulk movement with the flowing groundwater |
| $q_s$ | **Sources / sinks** | Mass or energy added or removed (wells, rivers, reactions) |

where $n$ is porosity, $C$ is concentration (or temperature), $\mathbf{v}$ is the seepage velocity vector, and $\mathbf{D}$ is the hydrodynamic dispersion tensor.

### Heat transport form

For heat, we replace concentration $C$ with temperature $T$ and account for **thermal storage in both water and solids**:

$$\underbrace{\frac{\partial}{\partial t}\bigl[n \rho_w c_w T + (1-n) \rho_s c_s T\bigr]}_{\text{bulk thermal storage}} = \underbrace{\nabla \cdot (\lambda_{bulk} \nabla T)}_{\text{conduction}} + \underbrace{\nabla \cdot (n \mathbf{D}_m \nabla T)}_{\text{mechanical dispersion}} - \underbrace{\nabla \cdot (n \rho_w c_w \mathbf{v} T)}_{\text{advection}} + q_E$$

The key difference from solute transport: heat is stored in **both phases** (water and solid), giving rise to **thermal retardation** ($R \approx 3.5$ for our Limmat Valley gravels, as derived in NB2).

### Flow vs. transport: a comparison

| | Flow equation | Transport equation |
|---|---|---|
| **State variable** | Hydraulic head $h$ | Temperature $T$ (or concentration $C$) |
| **What drives movement** | Hydraulic gradient $\nabla h$ | Groundwater velocity $\mathbf{v}$ (advection) + gradient $\nabla T$ (dispersion) |
| **What spreads it** | Hydraulic conductivity $K$ | Dispersion tensor $\mathbf{D}$ + thermal conductivity $\lambda$ |
| **Storage** | Specific storage $S_s$ | Bulk volumetric heat capacity $(\rho c)_{bulk}$ |
| **Coupling** | Independent | **One-directional:** flow → transport (velocity field) |

> 💡 **Transport is coupled but one-directional.** The flow model produces a velocity field that drives advection. Temperature changes do not affect the flow field — this is valid for small temperature differences ($\Delta T < 5$–10 °C). Density-dependent flow, where temperature significantly changes water density and thus hydraulic gradients, is a separate topic not covered here.

In [ ]:
# Checkpoint 1 — ADE Comprehension
create_multiple_choice('task_t03_checkpoint_1')

---

## 2 — Numerical Stability: Peclet and Courant Numbers

Transport is numerically **harder** than flow. The flow equation is purely diffusive (elliptic), while the ADE includes advection (hyperbolic character) — this makes it prone to oscillations and numerical dispersion if the grid or time step is too coarse.

Two dimensionless numbers control stability:

### Grid Peclet number

$$Pe_{grid} = \frac{v \cdot \Delta x}{D_L} \leq 2$$

This is a **cell-scale** requirement. It says: the advective transport across one cell must not exceed twice the dispersive spreading. Rearranging for maximum cell size:

$$\Delta x \leq \frac{2 \cdot D_L}{v} = 2 \cdot \alpha_L$$

since $D_L = \alpha_L \cdot v$ when molecular diffusion is negligible.

### Courant number

$$Cr = \frac{v \cdot \Delta t}{\Delta x} \leq 1$$

This is a **time-step** requirement. It says: a solute parcel should not cross more than one cell per time step.

### Worked example with Limmat Valley values

Using values from NB2: $v = 11$ m/d, $\alpha_L = 10$ m, $D_L = \alpha_L \cdot v = 110$ m$^2$/d.

| Criterion | Formula | Result |
|---|---|---|
| Max cell size | $\Delta x \leq 2 \alpha_L$ | $\leq 20$ m |
| Max time step (for $\Delta x = 20$ m) | $\Delta t \leq \Delta x / v$ | $\leq 1.8$ days |

Compare this to the flow model, where cell sizes of 50–100 m and time steps of days to weeks are common. **Transport demands finer spatial and temporal resolution.**

### TVD schemes and relaxation

The **TVD (Total Variation Diminishing)** advection scheme available in MODFLOW 6 is less sensitive to the grid Peclet constraint. It can handle $Pe_{grid} > 2$ without spurious oscillations, at the cost of introducing some numerical dispersion. In practice, TVD allows coarser grids than the classical $Pe \leq 2$ rule — but the Courant criterion still applies.

### Heat vs. solute stability

Heat transport has an additional advantage: **thermal conduction** through the solid matrix increases the effective dispersion coefficient beyond pure mechanical dispersion. This makes the grid Peclet number slightly more forgiving for heat than for conservative solutes.

In [ ]:
# Checkpoint 2 — Peclet Calculation
check_task_with_solution('task_t03_checkpoint_2')

---

## 3 — MODFLOW 6 GWE Packages

MODFLOW 6 implements heat transport through the **Groundwater Energy (GWE)** model. Like GWF, GWE is built from interchangeable packages — each handles one physical process or boundary type.

| Package | Purpose | Limmat Valley use |
|---|---|---|
| **EST** | Energy storage and transfer: porosity, density, heat capacity | Thermal retardation ($R \approx 3.5$) from $n_e$, $\rho_s$, $c_s$ |
| **ADV** | Advection scheme selection (upstream, TVD) | TVD for stability on our grid |
| **CND** | Conduction + mechanical dispersion: $\lambda_{bulk}$, $\alpha_L$, $\alpha_T$ | Thermal conductivity + dispersivity |
| **SSM** | Source-sink mixing: temperature of each GWF flux | RIV (Limmat 12.5 °C, Sihl 10.6 °C), RCH (9.8 °C), WEL (10.5 °C) |
| **CTP** | Constant temperature (Dirichlet BC) | If fixed-temperature boundaries are needed |
| **IC** | Initial temperature distribution | Background 10.5 °C |
| **OC** | Output control (what to save, when) | Temperature fields, energy budgets |

### How GWE couples to GWF

The GWE model receives the velocity field from GWF through either:
- A **GWF-GWE Exchange** (both models in the same simulation), or
- The **FMI** (Flow Model Interface) package, which reads a saved GWF flow solution from file

In both cases, **GWE never solves the flow equation** — it only solves the energy transport equation using velocities computed by GWF.

### Common misconceptions

| Misconception | Reality |
|---|---|
| "Transport needs hydraulic conductivity $K$" | No — GWE receives the velocity field from GWF. It never sees $K$. |
| "Porosity is inherited from GWF" | No — porosity must be re-specified in EST. GWF may use total porosity for storage; GWE needs effective porosity for transport. |
| "Heat dispersion = solute dispersion" | The mechanical part ($\alpha_L \cdot v$) is identical. But heat adds thermal conduction ($\lambda_{bulk}$) where solutes have molecular diffusion ($D_m^*$). |
| "Finer time steps are always better" | The Courant criterion sets a maximum $\Delta t$, but excessively fine time steps waste computation without improving accuracy. |

<details>
<summary><strong>Optional: GWE ↔ GWT Package Mapping</strong> (click to expand)</summary>

MODFLOW 6 offers two transport model types: **GWE** (energy/heat) and **GWT** (solute). Their package structure is nearly identical — the physics is the same, only the "diffusive" component differs.

| GWE package | GWT equivalent | What changes |
|---|---|---|
| **EST** (Energy Storage and Transfer) | **MST** (Mass Storage and Transfer) | Heat capacity & density → sorption ($K_d$), decay rates |
| **CND** (Conduction-Dispersion) | **DSP** (Dispersion) | Thermal conductivity $\lambda_{bulk}$ → molecular diffusion $D_m^*$ |
| **CTP** (Constant Temperature) | **CNC** (Constant Concentration) | Temperature value → concentration value |
| **ESL** (Energy Source/Loading) | **SRC** (Mass Source/Loading) | Energy rate [W] → mass rate [kg/s] |
| **ADV** (Advection) | **ADV** (Advection) | Identical — same scheme, same velocity field |
| **SSM** (Source-Sink Mixing) | **SSM** (Source-Sink Mixing) | Identical — associates temperature/concentration with each GWF stress |
| **IC** (Initial Conditions) | **IC** (Initial Conditions) | Temperature → concentration |
| **OC** (Output Control) | **OC** (Output Control) | Identical |

**Key insight:** Mechanical dispersion ($\alpha_L$, $\alpha_T$) is identical in both GWE and GWT — it describes velocity-driven spreading that does not depend on the transported quantity. Only the "molecular" component differs: thermal conduction ($\lambda$) for heat vs. molecular diffusion ($D_m^*$) for solutes.

This is why **calibrating dispersivity with heat** transfers directly to a solute model: $\alpha_L$ and $\alpha_T$ are the same; you only need to update the diffusion term and add species-specific reactions.

</details>

---

## 4 — Translating the Perceptual Model to GWE Packages

In NB2, we identified the key transport features of the Limmat Valley. Here is how each maps to a MODFLOW 6 GWE package:

| Perceptual feature (from NB2) | Value | GWE package | How it enters the model |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | **EST** | Porosity input; controls pore velocity and thermal retardation |
| Solid density $\rho_s$ / heat capacity $c_s$ | 2650 kg/m³ / 880 J/(kg·K) | **EST** | Together with $n_e$ → thermal retardation $R \approx 3.5$ |
| Bulk thermal conductivity $\lambda_{bulk}$ | ~1.65 W/(m·K) | **CND** | Conduction component of energy transport |
| Longitudinal dispersivity $\alpha_L$ | 5–20 m | **CND** | Mechanical dispersion component |
| Background GW temperature | 10.5 °C | **IC** | Initial temperature field |
| River Limmat temperature | 12.5 °C | **SSM** | Auxiliary temperature on RIV package |
| River Sihl temperature | 10.6 °C | **SSM** | Auxiliary temperature on RIV package |
| Areal recharge temperature | 9.8 °C | **SSM** | Auxiliary temperature on RCH package |
| Lateral inflow temperature | 10.5 °C | **SSM** | Auxiliary temperature on WEL package |
| Thermal retardation $R \approx 3.5$ | — | (computed by **EST**) | Derived from $n_e$, $\rho_s$, $c_s$, $\rho_w$, $c_w$ |

> 🤔 **Reflection:** The energy budget from NB2 showed Limmat infiltration as the dominant thermal input to the aquifer. In the model, the **SSM** package handles this automatically — it assigns the Limmat's temperature (12.5 °C) to every unit of water that the RIV package exchanges with the aquifer. No separate "thermal source" is needed; the coupling between GWF stresses and GWE temperatures is built into the framework.

In [ ]:
# Checkpoint 3 — Package Identification
create_multiple_choice('task_t03_checkpoint_3')

---

## Summary

In this notebook, you learned:

- The **advection-dispersion equation** extends the flow equation to transport: advection moves the front, dispersion spreads it, and sources/sinks add or remove mass or energy
- **Numerical stability** requires grid Peclet $\leq 2$ (max cell size $= 2\alpha_L$) and Courant $\leq 1$ (max time step $= \Delta x / v$) — transport needs finer grids and shorter time steps than flow
- MODFLOW 6 **GWE packages** (EST, ADV, CND, SSM, IC, OC) each handle one aspect of the energy transport equation
- The perceptual model from NB2 **maps directly** to GWE inputs: porosity and thermal properties → EST, dispersivity and conductivity → CND, temperatures of inflows → SSM

### What You're Taking Forward

| Concept | You'll use it in… |
|---|---|
| ADE / Peclet / Courant criteria | NB4: grid and time step choices for the transport model |
| GWE packages (EST, CND, ADV, SSM) | NB4: FloPy implementation of each package |
| Perceptual → GWE package mapping | NB4: parameter assignment from NB2 values |
| GWE ↔ GWT equivalence | NB8: switching from heat to solute transport |

### Documentation Resources

Bookmark these — you'll need them in NB4:

- **MODFLOW 6 GWE documentation:** [USGS MODFLOW 6 Description of Input and Output](https://modflow6.readthedocs.io/en/latest/)
- **FloPy documentation:** [FloPy — Python interface to MODFLOW](https://flopy.readthedocs.io/en/latest/)

---

## Next Steps

You now have the conceptual framework for heat transport modelling in MODFLOW 6. In the next step, you'll implement this as a working model.

**Continue to:** [Step 4 — Model Implementation](./4_model_implementation.ipynb) — Build the GWE model in FloPy

**Review if needed:** [Step 2 — Perceptual Model](./2_perceptual_model.ipynb) — Transport parameters and thermal boundary conditions